# Dynamic FRET: HMM and BVA

.. _hmm_bva_notebook:

This notebook demonstrates the two tttrlib tools for **sub-burst FRET dynamics**
on *simulated* data (no data files needed):

* :class:`~tttrlib.HMM` — a photon-by-photon Hidden Markov Model that infers the
  number of hidden FRET states, their emission (FRET) values and the transition
  rates between them.
* :class:`~tttrlib.BVA` — Burst Variance Analysis, a model-free test that
  separates static from dynamic species by comparing the per-burst spread of the
  proximity ratio against the shot-noise limit.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tttrlib

rng = np.random.default_rng(0)


## Simulate a two-state dynamic FRET dataset

We build a ground-truth HMM with two hidden states (a low-FRET and a
high-FRET conformation) that slowly interconvert, then generate photon streams
along random burst time axes with :func:`HMM.simulate_bursts`. Stream ``0`` is
the donor channel, stream ``1`` the acceptor.


In [ ]:
# ground-truth model: prior, one-tick transition matrix, emission (FRET) matrix
true = tttrlib.HmmModel(
    [0.5, 0.5],
    [0.9995, 0.0005, 0.0008, 0.9992],      # slow switching
    [0.80, 0.20,  0.25, 0.75],             # state 0: E~0.2, state 1: E~0.75
)

# random burst time axes (integer macro-times), ~400 photons/burst
n_bursts, burst_len = 80, 400
times = [np.cumsum(rng.integers(1, 60, size=burst_len)).astype(np.int64)
         for _ in range(n_bursts)]
streams = tttrlib.HMM.simulate_bursts(true, [list(t) for t in times], seed=1)

eng = tttrlib.HMM()
eng.set_bursts([list(map(int, t)) for t in times],
               [list(map(int, s)) for s in streams], n_streams=2)
print(eng.get_n_bursts(), 'bursts', eng.get_n_photons(), 'photons',
      len(eng.get_unique_dt()), 'unique dt')


## Fit H2MM and select the number of states by BIC

We fit 1, 2 and 3 state models (each with a few random restarts) and pick the
one with the lowest Bayesian Information Criterion. The two-state model should
win and recover the true emission matrix.


In [ ]:
fits = {k: eng.fit(k, n_restarts=3, max_iter=500, tol=1e-7, seed=0) for k in (1, 2, 3)}
for k, f in fits.items():
    print(f'{k} state(s): logL={f.loglik:9.1f}  BIC={f.bic():9.1f}  '
          f'({f.n_iter} EM maps, converged={f.converged})')

best = min(fits.values(), key=lambda m: m.bic())
print('\nselected states:', best.n_states())
# order states by donor probability for a stable comparison with the truth
order = np.argsort(best.obs_np[:, 0])[::-1]
print('recovered emission (FRET) matrix:\n', np.round(best.obs_np[order], 3))
print('true emission matrix:\n', np.round(true.obs_np, 3))


## Per-photon state path (Viterbi)

The Viterbi decoder assigns the most-likely hidden state to every photon. Here
we show the decoded state track for one burst.


In [ ]:
path, icl = eng.viterbi_path(best)
print('ICL =', round(icl, 1))

# states of the photons in the first burst
b0, b1 = 0, len(times[0])
fig, ax = plt.subplots(figsize=(8, 2))
ax.step(np.arange(b1 - b0), path[b0:b1], where='mid')
ax.set(xlabel='photon index in burst', ylabel='state', yticks=[0, 1],
       title='Viterbi state path (burst 0)')
plt.tight_layout()


## Burst Variance Analysis

BVA needs a :class:`~tttrlib.TTTR` photon stream. We build one in memory with two
channels (0 = donor, 1 = acceptor): a set of **static** bursts at a constant
proximity ratio and a set of **dynamic** bursts that switch between low and high
FRET within the burst. BVA should place the static bursts on the shot-noise line
and the dynamic bursts above it.


In [ ]:
def build_tttr(bursts):
    macro, chan, bounds = [], [], []
    t = 0
    for ch in bursts:
        start = len(macro)
        for c in ch:
            t += 1; macro.append(t); chan.append(int(c))
        t += 1_000_000  # dark gap between bursts
        bounds.append((start, len(macro)))
    d = tttrlib.TTTR()
    d.append_events(np.asarray(macro, np.uint64), np.zeros(len(macro), np.uint16),
                    np.asarray(chan, np.int8), np.zeros(len(macro), np.int8), False, 0)
    return d, np.asarray(bounds, np.int64)  # (n, 2) [start, stop]

n_slice = 5
static = [(rng.random(400) < 0.5).astype(int) for _ in range(200)]
dynamic = [np.concatenate([(rng.random(50) < (0.15 if rng.random() < .5 else 0.85)).astype(int)
                           for _ in range(8)]) for _ in range(200)]

res = {}
for name, b in [('static', static), ('dynamic', dynamic)]:
    d, bounds = build_tttr(b)
    bva = tttrlib.BVA(d); bva.set_donor([0]); bva.set_acceptor([1])
    bva.compute(bounds, n_slice)   # positional: (n, 2) bursts, photons/slice
    res[name] = (bva.proximity_ratio_mean, bva.proximity_ratio_std)


In [ ]:
p = np.linspace(0.01, 0.99, 200)
line_p, line_s = tttrlib.BVA.compute_static_bva_line(list(p), n_slice)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(line_p, line_s, 'k-', lw=2, label='shot-noise (static) line')
ax.scatter(*res['static'], s=14, alpha=.5, label='static bursts')
ax.scatter(*res['dynamic'], s=14, alpha=.5, label='dynamic bursts')
ax.set(xlabel='proximity ratio mean', ylabel='proximity ratio std',
       xlim=(0, 1), ylim=(0, 0.5), title='Burst Variance Analysis')
ax.legend()
plt.tight_layout()


The dynamic bursts sit clearly above the shot-noise line, while the static
population hugs it — the signature BVA uses to flag conformational dynamics. See
:ref:`hmm_bva_guide` for the full API and :ref:`hmm_performance` for how the
H2MM engine compares to the reference ``H2MM_C`` library.
